# Silver Layer - Packaged Water Marketing
## Purpose
This notebook transforms Bronze Delta data into a clean, validated, analytics-ready Silver dataset.

### Key Responsibilities
- Schema standardization
- Schema drift handling
- Deduplication
- Text normalization
- Business rule validation
- Derived metric generation
- Clean / Reject split
- Idempotent Delta write

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_PATH = "/Volumes/workspace/water_bottle_db/bronze/packaged_water_marketing"

SILVER_CLEAN_PATH = "/Volumes/workspace/water_bottle_db/silver/packaged_water_marketing_clean"
SILVER_REJECTS_PATH = "/Volumes/workspace/water_bottle_db/silver/packaged_water_marketing_rejects"
SILVER_RUNLOG_PATH = "/Volumes/workspace/water_bottle_db/silver/_runlog_packaged_water_marketing"

has  = lambda d, c: c in d.columns
col0 = lambda d, c: F.col(c) if has(d, c) else F.lit(None)

## Step 1 - Read Bronze and Capture Schema Drift
This step reads Bronze Delta data and protects the Silver pipeline from future source changes.

Any unexpected columns are preserved inside `extras_json` so the pipeline remains stable even if the API schema changes.

In [0]:
EXPECTED = [
    "transaction_id","record_id","raw_id",
    "brand","product_name","category","water_type","product_tier","bottle_size_ml",
    "mrp","selling_price","discount_percent","cost_price",
    "sales_units","marketing_spend",
    "avg_rating","ratings_count","reviews_count","seller_rating",
    "sales_channel","platform_source","seller_name","platform_url",
    "stock_status","delivery_days","distributor_count","retailer_count",
    "city","state","region",
    "source_type","activity_date","ingestion_timestamp",
    "bronze_ingestion_ts"
]

bronze_df = spark.read.format("delta").load(BRONZE_PATH)

if not has(bronze_df, "bronze_ingestion_ts"):
    bronze_df = bronze_df.withColumn(
        "bronze_ingestion_ts",
        F.coalesce(F.to_timestamp(col0(bronze_df, "ingestion_timestamp")), F.current_timestamp())
    )

for c in EXPECTED:
    if not has(bronze_df, c):
        bronze_df = bronze_df.withColumn(c, F.lit(None))

extra_cols = [c for c in bronze_df.columns if c not in set(EXPECTED) and c != "extras_json"]
extras = F.to_json(F.struct(*[F.col(c).cast("string").alias(c) for c in extra_cols])) if extra_cols else F.lit("{}")

silver_work_df = bronze_df.withColumn("extras_json", extras).select(*EXPECTED, "extras_json")
display(silver_work_df.limit(10))

transaction_id,record_id,raw_id,brand,product_name,category,water_type,product_tier,bottle_size_ml,mrp,selling_price,discount_percent,cost_price,sales_units,marketing_spend,avg_rating,ratings_count,reviews_count,seller_rating,sales_channel,platform_source,seller_name,platform_url,stock_status,delivery_days,distributor_count,retailer_count,city,state,region,source_type,activity_date,ingestion_timestamp,bronze_ingestion_ts,extras_json
3d02c4ac-0244-43c3-a36f-c6484b9430fa,28358.0,null,Bisleri,Bisleri 2000ml,Packaged Drinking Water,Mineral,Premium,2000.0,94.95,79.54,16.23,68.49,41.0,232.77,4.9,54.0,53.0,4.6,Online,Amazon,Seller_18,https://example.com/product,In Stock,1,1.0,84,Kolkata,West Bengal,East,API,2024-08-31,2024-08-31T07:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Festive"",""offer_type"":""None"",""promotion_flag"":""True""}"
7b69342c-a3e1-4006-957d-c8970d41bd45,27452.0,null,Vedica,Vedica 1000ml,Packaged Drinking Water,RO,Economy,1000.0,34.9,30.24,13.36,24.16,175.0,914.17,4.9,51.0,29.0,4.0,Online,Jiomart,Seller_19,https://example.com/product,Out of Stock,2,4.0,78,Kolkata,West Bengal,East,API,2024-08-29,2024-08-31T07:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Seasonal"",""offer_type"":""None"",""promotion_flag"":""True""}"
7e5b8c97-5846-4118-8cef-990a0293bbc2,233.0,null,Rail Neer,Rail Neer 200ml,Packaged Drinking Water,RO,Economy,200.0,11.67,11.6,0.6,8.28,115.0,212.46,4.7,113.0,33.0,4.2,Online,Jiomart,Seller_2,https://example.com/product,In Stock,1,1.0,118,Chennai,Tamil Nadu,South,API,2024-08-30,2024-08-31T07:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Seasonal"",""offer_type"":""None"",""promotion_flag"":""False""}"
82af6c59-db5e-43ab-be81-8e3587672b34,31726.0,null,Aquafina,Aquafina 500ml,Packaged Drinking Water,RO,Premium,500.0,44.09,35.22,20.11,27.78,192.0,381.11,4.9,284.0,116.0,4.3,Online,Amazon,Seller_5,https://example.com/product,In Stock,1,5.0,93,Bhopal,Madhya Pradesh,Central,API,2024-08-30,2024-08-31T07:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Flash Sale"",""offer_type"":""None"",""promotion_flag"":""True""}"
e123c10c-23f3-43f4-becb-48b703ddea52,45719.0,null,Kinley,Kinley 1000ml,Packaged Drinking Water,Alkaline,Mid,1000.0,48.97,41.92,14.39,35.19,157.0,927.06,4.9,159.0,57.0,3.8,Online,Blinkit,Seller_7,https://example.com/product,In Stock,4,3.0,89,Surat,Gujarat,West,API,2024-08-31,2024-08-31T07:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Seasonal"",""offer_type"":""None"",""promotion_flag"":""True""}"
12351efd-273c-4883-b2c6-a84f5d46b472,48421.0,null,Tata Copper+,Tata Copper+ 200ml,Packaged Drinking Water,Alkaline,Premium,200.0,24.74,22.13,10.54,15.59,78.0,189.45,3.6,289.0,248.0,3.6,Online,Amazon,Seller_1,https://example.com/product,In Stock,4,5.0,52,Ahmedabad,Gujarat,West,API,2024-08-29,2024-08-31T08:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Festive"",""offer_type"":""Cashback"",""promotion_flag"":""False""}"
5b06fcf6-3c5a-4b29-ae7f-2e9ea9f1cefc,410.0,null,Tata Copper+,Tata Copper+ 1000ml,Packaged Drinking Water,RO,Economy,1000.0,34.74,33.06,4.83,24.46,66.0,212.29,4.9,249.0,35.0,4.6,Online,Blinkit,Seller_1,https://example.com/product,In Stock,5,2.0,92,Chennai,Tamil Nadu,South,API,2024-08-30,2024-08-31T08:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Seasonal"",""offer_type"":""Cashback"",""promotion_flag"":""False""}"
731fee49-afe3-4cc3-a0a5-57246af26461,13356.0,null,Bailley,Bailley 1000ml,Packaged Drinking Water,Alkaline,Premium,1000.0,57.8,51.38,11.1,36.95,128.0,726.32,4.5,224.0,163.0,3.9,Online,Flipkart,Seller_16,https://example.com/product,In Stock,4,2.0,null,Pune,Maharashtra,West,API,2024-08-29,2024-08-31T08:00:00+00:00,2026-04-20T08:18:51.599Z,"{""campaign_type"":""Festive"",""offer_type"":""Discount"",""promotion_flag"":""True""}"
1cdccfbc-05b6-4b7a-84c1-7e3a47410f85,20899.0,null,Kinley,Kinley 1000ml,Packaged Drinking Water,Alkaline,Premium,1000.0,55.03,49.67,9.74,40.07,175.0,729.96,4.5,88.0,32.0,4.5,O

Reads Bronze data, ensures required baseline columns exist, and captures unknown incoming columns into extras_json

## Step 2 - Promote Important JSON Fields and Standardize Schema
This step extracts important marketing fields from `extras_json` if needed and safely casts all columns into a stable Silver schema.

In [0]:
def safe_to(col_expr, target):
    s = F.trim(col_expr.cast("string"))
    if target == "string":
        return s.cast("string")
    if target == "int":
        return F.when(
            s.rlike(r"^[+-]?\d+(\.0+)?$"),
            F.regexp_replace(s, r"\.0+$", "").cast("int")
        ).otherwise(F.lit(None).cast("int"))
    if target.startswith("decimal"):
        clean = F.regexp_replace(s, r"[^\d\.\-]", "")
        return F.when(
            clean.rlike(r"^[+-]?\d+(\.\d+)?$"),
            clean.cast(target)
        ).otherwise(F.lit(None).cast(target))
    if target == "date":
        return F.coalesce(
            F.to_date(s),
            F.to_date(s, "yyyy-MM-dd"),
            F.to_date(s, "MM/dd/yyyy"),
            F.to_date(s, "dd-MM-yyyy")
        )
    if target == "timestamp":
        return F.coalesce(
            F.to_timestamp(s),
            F.to_timestamp(s, "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(s, "yyyy-MM-dd'T'HH:mm:ss"),
            F.to_timestamp(s, "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            F.to_timestamp(s, "MM/dd/yyyy HH:mm:ss"),
            F.to_timestamp(s, "MM/dd/yyyy")
        )
    return col_expr.cast(target)

def from_col_or_extras(df, col_name):
    direct = F.trim(col0(df, col_name).cast("string"))
    from_json = F.get_json_object(col0(df, "extras_json").cast("string"), f"$.{col_name}")
    return F.when(direct.isNotNull() & (direct != ""), direct).otherwise(from_json)

Interpretation

Reusable helper functions for safe type conversion.

In [0]:
df = silver_work_df \
    .withColumn("campaign_type", from_col_or_extras(silver_work_df, "campaign_type")) \
    .withColumn("offer_type", from_col_or_extras(silver_work_df, "offer_type")) \
    .withColumn("promotion_flag", from_col_or_extras(silver_work_df, "promotion_flag"))

SPEC = {
    "transaction_id": "string", "record_id": "int", "raw_id": "int",
    "brand": "string", "product_name": "string", "category": "string",
    "water_type": "string", "product_tier": "string", "bottle_size_ml": "int",
    "campaign_type": "string", "offer_type": "string", "promotion_flag": "string",
    "mrp": "decimal(10,2)", "selling_price": "decimal(10,2)",
    "discount_percent": "decimal(5,2)", "cost_price": "decimal(10,2)",
    "sales_units": "int", "marketing_spend": "decimal(12,2)",
    "avg_rating": "decimal(3,2)", "ratings_count": "int", "reviews_count": "int",
    "seller_rating": "decimal(3,2)",
    "sales_channel": "string", "platform_source": "string", "seller_name": "string", "platform_url": "string",
    "stock_status": "string", "delivery_days": "int", "distributor_count": "int", "retailer_count": "int",
    "city": "string", "state": "string", "region": "string",
    "source_type": "string", "activity_date": "date", "ingestion_timestamp": "timestamp",
    "bronze_ingestion_ts": "timestamp", "extras_json": "string"
}

base = df
out = df
cast_err_exprs = []

for c, t in SPEC.items():
    if not has(out, c):
        out = out.withColumn(c, F.lit(None).cast(t))
    out = out.withColumn(c, safe_to(col0(out, c), t))

    cast_err_exprs.append(
        F.when(
            (F.trim(col0(base, c).cast("string")) != "") &
            col0(base, c).isNotNull() &
            F.col(c).isNull(),
            F.lit(c)
        )
    )

df_std = out.withColumn(
    "dq_error_reason",
    F.concat_ws("|", F.array_distinct(F.array_remove(F.array(*cast_err_exprs), F.lit(None))))
)

display(df_std.select(
    "transaction_id","record_id","activity_date","ingestion_timestamp",
    "campaign_type","promotion_flag","offer_type","extras_json","dq_error_reason"
).limit(20))

transaction_id,record_id,activity_date,ingestion_timestamp,campaign_type,promotion_flag,offer_type,extras_json,dq_error_reason
0fb120a1-ce6a-4253-be92-ab37609ccd42,5209,2024-01-01,2024-01-01T01:00:00.000Z,Seasonal,True,Discount,"{""campaign_type"":""Seasonal"",""offer_type"":""Discount"",""promotion_flag"":""True""}",
262eac26-05a7-49da-9e68-1842ab0a2789,33319,2024-01-01,2024-01-01T01:00:00.000Z,Flash Sale,False,Cashback,"{""campaign_type"":""Flash Sale"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",
1940b8ea-30f4-4c24-bbc4-ca5c195208ed,41953,2024-01-01,2024-01-01T01:00:00.000Z,Seasonal,False,BOGO,"{""campaign_type"":""Seasonal"",""offer_type"":""BOGO"",""promotion_flag"":""False""}",
98886f71-d94a-4b11-8b30-40f3d364665e,15662,2024-01-01,2024-01-01T01:00:00.000Z,Festive,False,Cashback,"{""campaign_type"":""Festive"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",
4b2ab296-5c65-4969-a1dc-f1c2974dfd0e,40221,2024-01-01,2024-01-01T01:00:00.000Z,Flash Sale,False,Cashback,"{""campaign_type"":""Flash Sale"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",
5007a46d-8c9a-44e3-a9bc-490e3ed2ee7a,22510,2024-01-01,2024-01-01T02:00:00.000Z,Always On,True,None,"{""campaign_type"":""Always On"",""offer_type"":""None"",""promotion_flag"":""True""}",
c07df1e6-a4a8-445e-8797-d31242eaea81,39435,2024-01-01,2024-01-01T03:00:00.000Z,Seasonal,True,Discount,"{""campaign_type"":""Seasonal"",""offer_type"":""Discount"",""promotion_flag"":""True""}",
306f6f8c-5fc5-46ba-9c63-137d3a42da68,28162,2024-01-01,2024-01-01T03:00:00.000Z,Seasonal,False,Cashback,"{""campaign_type"":""Seasonal"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",
a9c9a951-582d-4e70-acf6-3162bd5f97dd,39662,2024-01-01,2024-01-01T03:00:00.000Z,Seasonal,True,Discount,"{""campaign_type"":""Seasonal"",""offer_type"":""Discount"",""promotion_flag"":""True""}",
19e11c49-ec6c-4367-9e9e-7e9483859f3d,9594,2024-01-01,2024-01-01T03:00:00.000Z,Seasonal,False,Cashback,"{""campaign_type"":""Seasonal"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",


Extracts campaign_type, offer_type, and promotion_flag from either normal columns or extras_json, then safely casts all Silver columns to stable SSMS-friendly types.

## Step 3 - Deduplication and Promotion Normalization
This step keeps the latest record per `transaction_id`, creates normalized promotion fields, and flags critical identifier failures.

In [0]:
df = df_std.withColumn(
    "ingest_ts_eff",
    F.coalesce(F.col("ingestion_timestamp").cast("timestamp"),
               F.col("bronze_ingestion_ts").cast("timestamp"))
)

pf = F.upper(F.regexp_replace(F.trim(F.col("promotion_flag").cast("string")), r"\s+", ""))
df = df.withColumn(
    "promotion_flag_std",
    F.when(pf.isNull() | (pf == ""), "NO")
     .when(pf.isin("TRUE","1","YES","Y","T"), "YES")
     .when(pf.isin("FALSE","0","NO","N","F"), "NO")
     .otherwise("NO")
)

offer_clean = F.upper(F.regexp_replace(F.trim(F.col("offer_type").cast("string")), r"\s+", " "))
df = df.withColumn(
    "offer_type_std",
    F.when(F.col("promotion_flag_std") == "NO", "NONE")
     .when((F.col("promotion_flag_std") == "YES") & (offer_clean.isNull() | (offer_clean == "")), "UNKNOWN_OFFER")
     .otherwise(offer_clean)
)

crit_fail = (
    F.col("transaction_id").isNull() | (F.trim(F.col("transaction_id")) == "") |
    F.col("record_id").isNull() |
    F.col("activity_date").isNull() |
    F.col("ingest_ts_eff").isNull()
)

df = df.withColumn("dq_crit_fail", F.when(crit_fail, 1).otherwise(0)) \
       .withColumn(
           "dq_error_reason",
           F.concat_ws("|", F.array_distinct(F.array_remove(F.array(
               F.when(F.trim(F.col("dq_error_reason")) != "", F.col("dq_error_reason")).otherwise(F.lit(None)),
               F.when(F.col("dq_crit_fail") == 1, F.lit("CRITICAL_ID_OR_DATE_OR_INGEST_TS_MISSING")).otherwise(F.lit(None))
           ), F.lit(None))))
       )

w = Window.partitionBy("transaction_id").orderBy(
    F.col("ingest_ts_eff").desc_nulls_last(),
    F.col("bronze_ingestion_ts").desc_nulls_last()
)

df_silver_core = df.withColumn("_rn", F.row_number().over(w)) \
                   .filter(F.col("_rn") == 1) \
                   .drop("_rn") \
                   .withColumn("is_valid", F.col("dq_crit_fail") == 0)

display(df_silver_core.select(
    "transaction_id","record_id","activity_date","ingest_ts_eff",
    "promotion_flag_std","offer_type_std","is_valid","dq_error_reason"
).limit(20))

transaction_id,record_id,activity_date,ingest_ts_eff,promotion_flag_std,offer_type_std,is_valid,dq_error_reason
000264cc-5544-431f-8852-6d5d88a79355,41096,2024-03-17,2024-03-18T17:00:00.000Z,YES,UNKNOWN_OFFER,true,
000a5e1f-2f52-4f4d-b6b0-e8446d44fdb5,46460,2024-04-30,2024-05-02T09:00:00.000Z,NO,NONE,true,
000a9067-b5f0-4b99-8bc8-df6a7d9fe6a7,36942,2024-05-15,2024-05-16T08:00:00.000Z,YES,BOGO,true,
000ccbf5-05ab-4b45-9e46-437ad181f3d6,36953,2024-04-25,2024-04-25T17:00:00.000Z,YES,NONE,true,
000d64b2-92cb-440c-8eb7-3659714ea73b,20891,2024-02-20,2024-02-20T14:00:00.000Z,NO,NONE,true,
000eae96-a8f1-4654-9f8e-e7d31fdf8637,15847,2024-09-09,2024-09-10T10:00:00.000Z,NO,NONE,true,
000f2f1d-58df-418a-8828-76f47b810593,12131,2024-06-29,2024-06-30T12:00:00.000Z,YES,DISCOUNT,true,
00140e03-25ab-49e2-a6cc-d11c0ee5ad1e,17876,2024-12-15,2024-12-16T21:00:00.000Z,NO,NONE,true,
00146636-4f22-4bae-8b9c-1e83b36890e3,19326,2024-04-05,2024-04-06T04:00:00.000Z,NO,NONE,true,
001957a1-1a00-422f-9d56-93b5e0f5a45b,24177,2024-08-10,2024-08-10T13:00:00.000Z,NO,NONE,true,


Interpretation

Creates ingest_ts_eff, normalizes promo fields, validates critical identifiers, and keeps the latest row per transaction_id.

## Step 4 - Text Normalization and Brand Standardization
This step removes whitespace issues, standardizes casing, and creates a canonical `brand_std` field for reporting consistency.

In [0]:
df = df_silver_core

def clean_text(c):
    return F.regexp_replace(F.trim(c.cast("string")), r"\s+", " ")

text_cols = [
    "brand","product_name","category","water_type","product_tier",
    "campaign_type","offer_type","sales_channel","platform_source",
    "seller_name","stock_status","city","state","region","source_type"
]

for c in text_cols:
    if has(df, c):
        df = df.withColumn(c, clean_text(F.col(c)))

df = df.withColumn("city", F.when(clean_text(F.col("city")) == "", "UNKNOWN").otherwise(F.initcap(F.col("city")))) \
       .withColumn("state", F.when(clean_text(F.col("state")) == "", "UNKNOWN").otherwise(F.initcap(F.col("state")))) \
       .withColumn("region", F.when(clean_text(F.col("region")) == "", "UNKNOWN").otherwise(F.initcap(F.col("region")))) \
       .withColumn("stock_status", F.upper(F.col("stock_status"))) \
       .withColumn("source_type", F.upper(F.col("source_type")))

b = F.lower(clean_text(F.col("brand")))
df_silver_text_std = df.withColumn(
    "brand_std",
    F.when(b.rlike(r"\baquafina\b"), "AQUAFINA")
     .when(b.rlike(r"\bbisleri\b"), "BISLERI")
     .when(b.rlike(r"\brail\s*neer\b|\brailneer\b"), "RAIL NEER")
     .when(b.rlike(r"\bkinley\b"), "KINLEY")
     .when(b.rlike(r"\bbailley\b|\bbailey\b"), "BAILLEY")
     .when(b.rlike(r"\bvedica\b"), "VEDICA")
     .when(b.rlike(r"\bhimalayan\b"), "HIMALAYAN")
     .when(b.rlike(r"\btata\b.*\bcopper\+?\b|\bcopper\+\b"), "TATA COPPER+")
     .otherwise(F.upper(F.col("brand")))
)

display(df_silver_text_std.select("brand","brand_std","city","state","region").limit(20))

brand,brand_std,city,state,region
Bailley,BAILLEY,Jaipur,Rajasthan,North
Kinley,KINLEY,Ahmedabad,Gujarat,West
Himalayan,HIMALAYAN,Indore,Madhya Pradesh,Central
Himalayan,HIMALAYAN,Indore,Madhya Pradesh,Central
Bailley,BAILLEY,Mumbai,Maharashtra,West
Vedica,VEDICA,Pune,Maharashtra,West
Vedica,VEDICA,Hyderabad,Telangana,South
Kinley,KINLEY,Delhi,Delhi,North
Kinley,KINLEY,Delhi,Delhi,North
bailley,BAILLEY,Mumbai,Maharashtra,West


Interpretation

Normalizes text formatting and creates standardized brand values.

## Step 5 - Validation Rules
This step applies business quality checks for seller, ratings, and pricing. Invalid records are flagged using `dq_error_reason`.

In [0]:
df = df_silver_text_std

def add_reason(df, cond, reason):
    return df.withColumn(
        "dq_error_reason",
        F.concat_ws("|", F.array_distinct(F.array_remove(F.array(
            F.when(F.trim(F.col("dq_error_reason")) != "", F.col("dq_error_reason")).otherwise(F.lit(None)),
            F.when(cond, F.lit(reason)).otherwise(F.lit(None))
        ), F.lit(None))))
    )

df = df.withColumn(
    "seller_name",
    F.when(F.col("seller_name").isNull() | (F.trim(F.col("seller_name")) == ""), "UNKNOWN").otherwise(F.col("seller_name"))
).withColumn(
    "seller_rating",
    F.when(F.upper(F.col("seller_name")) == "UNKNOWN", F.lit(None).cast("decimal(3,2)"))
     .otherwise(F.col("seller_rating").cast("decimal(3,2)"))
)

bad_seller_rating = F.col("seller_rating").isNotNull() & ((F.col("seller_rating") < 0) | (F.col("seller_rating") > 5))
bad_avg = F.col("avg_rating").isNotNull() & ((F.col("avg_rating") < 0) | (F.col("avg_rating") > 5))
bad_counts_neg = ((F.col("ratings_count") < 0) | (F.col("reviews_count") < 0))
bad_order = F.col("ratings_count").isNotNull() & F.col("reviews_count").isNotNull() & (F.col("ratings_count") < F.col("reviews_count"))
bad_pos = ((F.col("mrp") <= 0) | (F.col("selling_price") <= 0) | (F.col("cost_price") <= 0))
bad_sell_gt_mrp = F.col("selling_price") > F.col("mrp")
bad_cost_gt_sell = F.col("cost_price") > F.col("selling_price")

df = add_reason(df, bad_seller_rating, "SELLER_RATING_OUT_OF_RANGE")
df = add_reason(df, bad_avg, "AVG_RATING_OUT_OF_RANGE")
df = add_reason(df, bad_counts_neg, "RATING_COUNTS_NEGATIVE")
df = add_reason(df, bad_order, "RATINGS_LT_REVIEWS")
df = add_reason(df, bad_pos, "PRICE_NON_POSITIVE")
df = add_reason(df, bad_sell_gt_mrp, "SELLING_GT_MRP")
df = add_reason(df, bad_cost_gt_sell, "COST_GT_SELLING")

df_silver_validated = df.withColumn(
    "is_valid",
    F.when(F.trim(F.col("dq_error_reason")) != "", F.lit(False)).otherwise(F.lit(True))
)

display(df_silver_validated.select(
    "transaction_id","brand_std","mrp","selling_price","cost_price","is_valid","dq_error_reason"
).limit(20))

transaction_id,brand_std,mrp,selling_price,cost_price,is_valid,dq_error_reason
000264cc-5544-431f-8852-6d5d88a79355,BAILLEY,21.67,21.25,16.01,true,
000a5e1f-2f52-4f4d-b6b0-e8446d44fdb5,KINLEY,89.26,67.63,60.05,true,
000a9067-b5f0-4b99-8bc8-df6a7d9fe6a7,HIMALAYAN,15.89,14.29,9.93,true,
000ccbf5-05ab-4b45-9e46-437ad181f3d6,HIMALAYAN,75.85,73.26,44.18,true,
000d64b2-92cb-440c-8eb7-3659714ea73b,BAILLEY,22.54,17.73,13.62,true,
000eae96-a8f1-4654-9f8e-e7d31fdf8637,VEDICA,76.29,75.60,48.51,true,
000f2f1d-58df-418a-8828-76f47b810593,VEDICA,63.26,57.16,42.45,true,
00140e03-25ab-49e2-a6cc-d11c0ee5ad1e,KINLEY,80.78,64.14,56.80,true,
00146636-4f22-4bae-8b9c-1e83b36890e3,KINLEY,31.91,27.49,23.01,true,
001957a1-1a00-422f-9d56-93b5e0f5a45b,BAILLEY,11.11,9.56,7.38,true,


Interpretation

Applies all business validation rules and finalizes is_valid.

## Step 6 - Derived Metrics
This step creates KPI-ready derived measures such as discount, margin, revenue, cost, and profit.

In [0]:
df = df_silver_validated

def safe_div(n, d):
    return F.when(d.isNull() | (d == 0), F.lit(None)).otherwise(n / d)

bad_su = F.col("sales_units").isNotNull() & (F.col("sales_units") < 0)

df = df.withColumn(
    "dq_error_reason",
    F.concat_ws("|", F.array_distinct(F.array_remove(F.array(
        F.when(F.trim(F.col("dq_error_reason")) != "", F.col("dq_error_reason")).otherwise(F.lit(None)),
        F.when(bad_su, F.lit("SALES_UNITS_NEGATIVE")).otherwise(F.lit(None))
    ), F.lit(None))))
).withColumn(
    "sales_units",
    F.when(bad_su, F.lit(None).cast("int")).otherwise(F.col("sales_units").cast("int"))
)

mrp = F.col("mrp").cast("decimal(10,2)")
sp  = F.col("selling_price").cast("decimal(10,2)")
cp  = F.col("cost_price").cast("decimal(10,2)")
su  = F.col("sales_units").cast("int")

df_silver_metrics = df \
    .withColumn("discount_amount", (mrp - sp).cast("decimal(10,2)")) \
    .withColumn("discount_percent_calc", (safe_div((mrp - sp), mrp) * 100).cast("decimal(5,2)")) \
    .withColumn("unit_profit", (sp - cp).cast("decimal(10,2)")) \
    .withColumn("unit_margin_pct", (safe_div((sp - cp), sp) * 100).cast("decimal(6,2)")) \
    .withColumn("gross_revenue", (sp * su).cast("decimal(18,2)")) \
    .withColumn("total_cost", (cp * su).cast("decimal(18,2)")) \
    .withColumn("gross_profit", (F.col("gross_revenue") - F.col("total_cost")).cast("decimal(18,2)")) \
    .withColumn("distributor_count", F.coalesce(F.col("distributor_count").cast("int"), F.lit(0))) \
    .withColumn("retailer_count", F.coalesce(F.col("retailer_count").cast("int"), F.lit(0))) \
    .withColumn("is_valid", F.when(F.trim(F.col("dq_error_reason")) != "", F.lit(False)).otherwise(F.lit(True)))

display(df_silver_metrics.select(
    "transaction_id","brand_std","discount_amount","gross_revenue","gross_profit","is_valid","dq_error_reason"
).limit(20))

transaction_id,brand_std,discount_amount,gross_revenue,gross_profit,is_valid,dq_error_reason
000264cc-5544-431f-8852-6d5d88a79355,BAILLEY,0.42,4207.50,1037.52,true,
000a5e1f-2f52-4f4d-b6b0-e8446d44fdb5,KINLEY,21.63,3449.13,386.58,true,
000a9067-b5f0-4b99-8bc8-df6a7d9fe6a7,HIMALAYAN,1.60,1429.00,436.00,true,
000ccbf5-05ab-4b45-9e46-437ad181f3d6,HIMALAYAN,2.59,11062.26,4391.08,true,
000d64b2-92cb-440c-8eb7-3659714ea73b,BAILLEY,4.81,2269.44,526.08,true,
000eae96-a8f1-4654-9f8e-e7d31fdf8637,VEDICA,0.69,12398.40,4442.76,true,
000f2f1d-58df-418a-8828-76f47b810593,VEDICA,6.10,11260.52,2897.87,true,
00140e03-25ab-49e2-a6cc-d11c0ee5ad1e,KINLEY,16.64,9621.00,1101.00,true,
00146636-4f22-4bae-8b9c-1e83b36890e3,KINLEY,4.42,3766.13,613.76,true,
001957a1-1a00-422f-9d56-93b5e0f5a45b,BAILLEY,1.55,468.44,106.82,true,


Interpretation

Builds all core business measures required for Gold and BI.

## Step 7 - Final Standardization and Idempotent Silver Write
This step finalizes `event_date`, creates `record_hash`, splits valid/reject rows, and writes to Delta using idempotent MERGE logic.

In [0]:
df = df_silver_metrics

def clean_text(c):
    return F.regexp_replace(F.trim(c.cast("string")), r"\s+", " ")

ss = F.upper(clean_text(F.col("stock_status")))
df = df.withColumn(
    "stock_status",
    F.when(ss.isNull() | (ss == ""), "UNKNOWN")
     .when(ss.rlike(r"(IN\s*STOCK|AVAILABLE|YES|TRUE|1)"), "IN_STOCK")
     .when(ss.rlike(r"(OUT\s*OF\s*STOCK|OOS|NOT\s*AVAILABLE|NO|FALSE|0)"), "OUT_OF_STOCK")
     .otherwise("UNKNOWN")
)

st = F.upper(clean_text(F.col("source_type")))
df = df.withColumn("source_type", F.when(st.isin("API","FILE","MANUAL"), st).otherwise("UNKNOWN"))

df = df.withColumn(
    "event_date",
    F.coalesce(
        F.col("activity_date").cast("date"),
        F.to_date(F.col("ingestion_timestamp").cast("timestamp")),
        F.to_date(F.col("bronze_ingestion_ts").cast("timestamp"))
    )
).withColumn("event_year", F.year("event_date").cast("int")) \
 .withColumn("event_month", F.month("event_date").cast("int")) \
 .withColumn("event_day", F.dayofmonth("event_date").cast("int")) \
 .withColumn("silver_ingestion_ts", F.current_timestamp())

brand_col = F.col("brand_std") if "brand_std" in df.columns else F.col("brand")
hash_expr = F.concat_ws("||",
    F.coalesce(brand_col.cast("string"), F.lit("")),
    F.coalesce(F.col("product_name").cast("string"), F.lit("")),
    F.coalesce(F.col("bottle_size_ml").cast("string"), F.lit("")),
    F.coalesce(F.col("platform_source").cast("string"), F.lit("")),
    F.coalesce(F.col("seller_name").cast("string"), F.lit("")),
    F.coalesce(F.col("event_date").cast("string"), F.lit(""))
)

df = df.withColumn("record_hash", F.sha2(hash_expr, 256)) \
       .withColumn("is_valid", F.coalesce(F.col("is_valid").cast("boolean"), F.lit(False)))

clean_df = df.filter(F.col("is_valid") == True).withColumn(
    "_merge_ts",
    F.coalesce(F.col("ingestion_timestamp"), F.col("bronze_ingestion_ts"), F.col("silver_ingestion_ts"))
)

rejects_df = df.filter(F.col("is_valid") == False).withColumn(
    "_merge_ts",
    F.coalesce(F.col("ingestion_timestamp"), F.col("bronze_ingestion_ts"), F.col("silver_ingestion_ts"))
)

def merge_upsert(path, src_df, key_col="transaction_id"):
    if not DeltaTable.isDeltaTable(spark, path):
        src_df.drop("_merge_ts").write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
        return

    tgt = DeltaTable.forPath(spark, path)
    cond = f"t.{key_col} = s.{key_col}"

    (tgt.alias("t")
        .merge(src_df.alias("s"), cond)
        .whenMatchedUpdateAll(condition="s._merge_ts >= t.ingestion_timestamp OR t.ingestion_timestamp IS NULL")
        .whenNotMatchedInsertAll()
        .execute()
    )

merge_upsert(SILVER_CLEAN_PATH, clean_df)
merge_upsert(SILVER_REJECTS_PATH, rejects_df)

print("✅ SILVER MERGE COMPLETE (Idempotent)")
print("Clean distinct transaction_id:", spark.read.format("delta").load(SILVER_CLEAN_PATH).select("transaction_id").distinct().count())
print("Reject distinct transaction_id:", spark.read.format("delta").load(SILVER_REJECTS_PATH).select("transaction_id").distinct().count())

# ============================================
# SILVER RUN AUDIT LOG
# ============================================

RUNLOG_PATH = "/Volumes/workspace/water_bottle_db/silver/_runlog_packaged_water_marketing"

clean_count = spark.read.format("delta").load(SILVER_CLEAN_PATH).count()
reject_count = spark.read.format("delta").load(SILVER_REJECTS_PATH).count()

runlog_df = spark.createDataFrame(
    [(clean_count, reject_count, clean_count + reject_count, "SILVER")],
    ["clean_rows", "reject_rows", "total_rows", "pipeline_stage"]
).withColumn("run_timestamp", F.current_timestamp())

runlog_df.write.format("delta").mode("append").save(RUNLOG_PATH)

print("✅ Silver run log updated")

✅ SILVER MERGE COMPLETE (Idempotent)
Clean distinct transaction_id: 48000
Reject distinct transaction_id: 0
✅ Silver run log updated


Interpretation

This is the production write cell. It prevents duplicate rows on reruns and makes the pipeline automation-safe.

# Silver Layer – Packaged Water Marketing Data Pipeline

## Overview
This notebook implements the **Silver layer** of the Medallion Architecture for the packaged water marketing dataset.  
The goal of this layer is to transform the raw Bronze data into a **clean, standardized, and analytics-ready dataset** that can safely feed the Gold analytical model and BI dashboards.

The Silver pipeline is designed to be **robust, automation-safe, and schema-drift tolerant**, ensuring the pipeline continues running even if the upstream dataset changes.

---

## Key Responsibilities of the Silver Layer

### 1. Schema Standardization
All incoming columns are cast into **stable, SSMS-friendly data types** to prevent schema inconsistencies in downstream systems.

### 2. Schema Drift Handling
If new columns appear in the source dataset, they are captured in an **`extras_json`** column instead of breaking the pipeline.

### 3. Data Cleaning & Normalization
Text fields are standardized using trimming, whitespace normalization, and proper casing.  
Important categorical fields such as **brand, region, and stock status** are normalized.

### 4. Deduplication
Duplicate records are removed using **`transaction_id`** as the primary identifier.  
The latest record is selected using **ingestion timestamps**.

### 5. Business Rule Validation
Multiple data quality validations are applied including:

- Pricing validation (MRP, selling price, cost price)
- Rating validation (0–5 scale)
- Ratings vs reviews consistency
- Seller information validation
- Sales units validation

Invalid records are flagged with **`dq_error_reason`**.

### 6. Derived Metric Generation
Additional analytical fields are generated including:

- `discount_amount`
- `discount_percent_calc`
- `unit_profit`
- `unit_margin_pct`
- `gross_revenue`
- `total_cost`
- `gross_profit`

These metrics form the foundation for downstream KPI calculations in the Gold layer.

### 7. Standardized Event Date
A unified **`event_date`** column is created using:

1. `activity_date`
2. `ingestion_timestamp`
3. `bronze_ingestion_ts`

This ensures time-based analytics remain stable.

### 8. Record Hash Generation
A **`record_hash`** column is generated using key business attributes to help maintain data consistency and traceability.

### 9. Clean / Reject Data Separation
Records are split into two outputs:

- **Clean dataset**
- **Reject dataset**

Invalid records remain traceable without affecting analytics.

### 10. Idempotent Delta Writes
Final outputs are written to Delta tables using **MERGE operations**, ensuring the pipeline remains safe for scheduled automated runs without creating duplicates.

---

## Output Tables

### Silver Clean Dataset
```
/Volumes/workspace/water_bottle_db/silver/packaged_water_marketing_clean
```

Contains validated and analytics-ready data.

### Silver Reject Dataset
```
/Volumes/workspace/water_bottle_db/silver/packaged_water_marketing_rejects
```

Contains records that failed validation rules.

---

## Downstream Usage

The Silver dataset acts as the **foundation for the Gold layer**, where the following will be implemented:

- Dimensional star schema
- Fact marketing tables
- Business KPI calculations
- Power BI dashboard reporting

The Silver layer ensures that all downstream analytics operate on **clean, reliable, and standardized data**.

In [0]:
df  = spark.read.format("delta").load(SILVER_CLEAN_PATH)
display(df)

transaction_id,record_id,raw_id,brand,product_name,category,water_type,product_tier,bottle_size_ml,mrp,selling_price,discount_percent,cost_price,sales_units,marketing_spend,avg_rating,ratings_count,reviews_count,seller_rating,sales_channel,platform_source,seller_name,platform_url,stock_status,delivery_days,distributor_count,retailer_count,city,state,region,source_type,activity_date,ingestion_timestamp,bronze_ingestion_ts,extras_json,campaign_type,offer_type,promotion_flag,dq_error_reason,ingest_ts_eff,promotion_flag_std,offer_type_std,dq_crit_fail,is_valid,brand_std,discount_amount,discount_percent_calc,unit_profit,unit_margin_pct,gross_revenue,total_cost,gross_profit,event_date,event_year,event_month,event_day,silver_ingestion_ts,record_hash
001e8d9c-4160-4d9a-a574-34e8bad8fac2,4977,null,Aquafina,Aquafina 500ml,Packaged Drinking Water,RO,Mid,500,39.33,36.94,6.07,25.49,67,218.79,3.60,96,79,3.70,Online,Blinkit,Seller_5,https://example.com/product,IN_STOCK,4,2,101,Bengaluru,Karnataka,South,API,2024-11-30,2024-12-01T02:00:00.000Z,2026-03-13T05:41:18.860Z,"{""campaign_type"":""Always On"",""offer_type"":""Cashback"",""promotion_flag"":""False""}",Always On,Cashback,False,,2024-12-01T02:00:00.000Z,NO,NONE,0,true,AQUAFINA,2.39,6.08,11.45,31.00,2474.98,1707.83,767.15,2024-11-30,2024,11,30,2026-03-16T12:41:20.101Z,572b4c533ccbe02fdd9e39dedef6a82df071ce825e7384c87b5a4864bff3f30e
00291146-01dc-4ab5-b2e2-cb40f01a5598,26521,null,Rail Neer,Rail Neer 200ml,Packaged Drinking Water,Spring,Economy,200,12.49,9.37,24.97,8.25,149,194.27,4.80,241,155,4.40,Online,Bigbasket,Seller_5,https://example.com/product,IN_STOCK,2,3,40,Kolkata,West Bengal,East,API,2024-10-27,2024-10-27T01:00:00.000Z,2026-03-11T14:13:27.278Z,"{""campaign_type"":""Flash Sale"",""offer_type"":""BOGO"",""promotion_flag"":""False""}",Flash Sale,BOGO,False,,2024-10-27T01:00:00.000Z,NO,NONE,0,true,RAIL NEER,3.12,24.98,1.12,11.95,1396.13,1229.25,166.88,2024-10-27,2024,10,27,2026-03-16T12:41:20.101Z,fac2545e335f8ee10e6700bbda1dbc0f22c598f20481db4a5f6d2b4e47ca8434
0031fd3e-24bc-4d84-8573-4b859913acfa,4208,null,Vedica,Vedica 200ml,Packaged Drinking Water,Alkaline,Mid,200,20.34,18.98,6.70,11.30,69,155.49,3.70,144,66,4.60,Online,Amazon,Seller_7,https://example.com/product,IN_STOCK,5,3,38,Bengaluru,Karnataka,South,API,2024-10-30,2024-10-30T21:00:00.000Z,2026-03-11T14:13:39.951Z,"{""campaign_type"":""Festive"",""offer_type"":""Cashback"",""promotion_flag"":""True""}",Festive,Cashback,True,,2024-10-30T21:00:00.000Z,YES,CASHBACK,0,true,VEDICA,1.36,6.69,7.68,40.46,1309.62,779.70,529.92,2024-10-30,2024,10,30,2026-03-16T12:41:20.101Z,c09688cec9c047ee3a2db6a608068c5c05664ae3e460dcf97350dda6b2ed489a
0074e163-dd74-4179-b0e1-5d7a3cf7762f,48593,null,Aquafina,Aquafina 500ml,Packaged Drinking Water,Mineral,Premium,500,38.65,32.83,15.05,23.35,110,521.56,4.70,81,65,4.00,Online,Jiomart,Seller_17,https://example.com/product,IN_STOCK,2,4,34,Ahmedabad,Gujarat,West,API,2024-10-11,2024-10-12T11:00:00.000Z,2026-03-11T14:12:58.640Z,"{""campaign_type"":""Always On"",""offer_type"":""Discount"",""promotion_flag"":""True""}",Always On,Discount,True,,2024-10-12T11:00:00.000Z,YES,DISCOUNT,0,true,AQUAFINA,5.82,15.06,9.48,28.88,3611.30,2568.50,1042.80,2024-10-11,2024,10,11,2026-03-16T12:41:20.101Z,9631b067f4bd066cb869b4d6308d354a7f83100f22164816648227f4bb44b41f
008837a4-f939-4bb8-ac14-1976ff3d55f0,36052,null,Himalayan,Himalayan 200ml,Packaged Drinking Water,RO,Mid,200,23.78,20.74,12.80,15.40,151,395.24,4.30,266,213,4.10,Online,Blinkit,Seller_3,https://example.com/product,IN_STOCK,4,1,101,Indore,Madhya Pradesh,Central,API,2024-11-23,2024-11-24T16:00:00.000Z,2026-03-11T14:14:59.510Z,"{""campaign_type"":""Seasonal"",""offer_type"":""Discount"",""promotion_flag"":""False""}",Seasonal,Discount,False,,2024-11-24T16:00:00.000Z,NO,NONE,0,true,HIMALAYAN,3.04,12.78,5.34,25.75,3131.74,2325.40,806.34,2024-11-23,2024,11,23,2026-03-16T12:41:20.101Z,f785ec8d9e9f84af2d8c676d57ec6cdf33664c442f5f0a02c22940fb1db5ef7d
0091a668-160b-4139-a